# 06 - QLoRA (Qwen3-1.7B) and zero-shot. The smoke test decides QLORA_BASE and QLORA_STACK.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import models
SMOKE = True
with evaluate.log_run("qwen3-qlora", 0, "smoke" if SMOKE else "----", is_synthetic=SYN) as rec:
    r = models.qlora_classifier(X, y, split, seed=0, smoke=SMOKE)
    rec["max_memory_gb"] = r["max_memory_gb"]
if not SMOKE:
    evaluate.save_scores("qwen3-qlora", 0, r["scores"], is_synthetic=SYN)   # repeat for SEEDS_FINAL
    evaluate.lock_from_val_a("qwen3-qlora", y, {"r": 16, "lr": 2e-4}, is_synthetic=SYN)
rec

In [ ]:
# Zero-shot: one fixed prompt, no edits after seeing any result
zs = {sp: models.zero_shot_scores(X.loc[split == sp, "text"].head(64 if SMOKE else None))
      for sp in ("val_a", "val_b", "test")}
if not SMOKE:
    evaluate.save_scores("zero-shot", 0, zs, is_synthetic=SYN)
    evaluate.lock_from_val_a("zero-shot", y, {"model": config.ZERO_SHOT_MODEL}, is_synthetic=SYN,
                             prompt_path=config.ZERO_SHOT_PROMPT)